# 01 Candidate Generation — Соусы

Цель: построить baseline-список пар-кандидатов внутри категории «Соусы» для дальнейшей ручной разметки. На этом шаге используем только title similarity; brand и weight/pack добавляются как признаки пары и не служат фильтром при генерации кандидатов.


## План

1. Найти локальный DuckDB-куб и загрузить `mpstats_products` по категории `Соусы`/`Соус`.
2. Подготовить один product-record на пару `marketplace + Артикул`: месяцы одного marketplace агрегируются, разные marketplace остаются отдельными записями.
3. Сгенерировать пары по token-overlap/Jaccard/fuzzy-like baseline из `research.dedup`.
4. Отметить cross-marketplace pairs и hard-negative candidates: тот же brand и близкий вес/pack, но разные вкусовые токены.
5. Сохранить CSV для следующего ноутбука и проверить долю внутри-/межмаркетплейсных пар.


In [ ]:
from pathlib import Path
import os
import sys
import time

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.dedup import CandidateGenerationConfig, generate_candidate_pairs, prepare_product_records

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 140)


In [ ]:
TARGET_CATEGORY = "Соусы"
CATEGORY_ALIASES = ["Соусы", "Соус"]
PRODUCTS_TABLE = "mpstats_products"
DATA_DIR = PROJECT_ROOT / "research" / "dedup" / "data"
CANDIDATES_PATH = DATA_DIR / "candidates_sauces.csv"

CANDIDATE_COLUMNS = [
    "raw_record_id_a",
    "raw_record_id_b",
    "marketplace_a",
    "marketplace_b",
    "marketplaces_a",
    "marketplaces_b",
    "sku_a",
    "sku_b",
    "title_a",
    "title_b",
    "brand_a",
    "brand_b",
    "unit_amount_a",
    "unit_amount_b",
    "total_amount_a",
    "total_amount_b",
    "multipack_count_a",
    "multipack_count_b",
    "baseline_similarity_score",
    "is_cross_marketplace_pair",
    "is_hard_negative_candidate",
]

candidate_config = CandidateGenerationConfig(
    min_similarity=0.40,
    min_shared_tokens=1,
    max_block_size=120,
    max_pair_candidates_for_scoring=120_000,
    max_candidates=None,
)


def resolve_duckdb_path(project_root: Path) -> Path:
    env_path = os.environ.get("MPSTATS_DUCKDB_PATH")
    candidates = [Path(env_path).expanduser() if env_path else None, project_root / "mpstats.duckdb"]
    for candidate in candidates:
        if candidate is not None and candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "DuckDB-куб не найден. Задайте MPSTATS_DUCKDB_PATH=/absolute/path/to/mpstats.duckdb "
        "или положите mpstats.duckdb в корень проекта."
    )


DB_PATH = resolve_duckdb_path(PROJECT_ROOT)
print(f"DuckDB cube: {DB_PATH}")


In [ ]:
with duckdb.connect(str(DB_PATH), read_only=True) as con:
    available_categories = con.execute(
        f'SELECT DISTINCT "Категория" FROM {PRODUCTS_TABLE} ORDER BY 1'
    ).fetchdf()["Категория"].dropna().tolist()
    real_category = next((category for category in CATEGORY_ALIASES if category in available_categories), None)
    if real_category is None:
        raise ValueError(f"Категория {TARGET_CATEGORY!r} не найдена. Доступно: {available_categories[:20]}")
    products_df = con.execute(
        f'SELECT * FROM {PRODUCTS_TABLE} WHERE "Категория" = ?',
        [real_category],
    ).fetchdf()

print(f"Resolved category: {real_category}")
print(f"Loaded rows: {len(products_df):,}")
display(products_df.head(3))


In [ ]:
product_records = prepare_product_records(products_df, candidate_config)
articles_by_marketplaces = product_records.groupby("sku")["marketplace"].nunique(dropna=True)
summary = pd.DataFrame(
    [
        {
            "raw_rows": len(products_df),
            "product_records": len(product_records),
            "unique_articles": product_records["sku"].nunique(dropna=True),
            "marketplaces": product_records["marketplace"].nunique(dropna=True),
            "articles_seen_in_multiple_marketplaces": int((articles_by_marketplaces > 1).sum()),
            "unique_titles": product_records["title_norm"].nunique(dropna=True),
            "brand_fill_share": product_records["brand_norm"].ne("").mean(),
            "multipack_gt_1_share": (pd.to_numeric(product_records["multipack_count"], errors="coerce") > 1).mean(),
        }
    ]
)
display(summary)
display(product_records.head(5))


In [ ]:
started_at = time.perf_counter()
candidates = generate_candidate_pairs(products_df, candidate_config)
elapsed_sec = time.perf_counter() - started_at

print(f"Candidate pairs: {len(candidates):,}")
print(f"Generation time: {elapsed_sec:.2f} sec")
display(candidates.head(10))


In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
candidates_to_save = candidates[CANDIDATE_COLUMNS].copy()
candidates_to_save.to_csv(CANDIDATES_PATH, index=False)

print(f"Saved candidates: {CANDIDATES_PATH}")
print(f"Rows saved: {len(candidates_to_save):,}")
display(candidates_to_save.head(5))

pair_scope_stats = (
    candidates_to_save.assign(
        pair_scope=candidates_to_save["is_cross_marketplace_pair"].map(
            {True: "cross_marketplace", False: "same_marketplace"}
        )
    )["pair_scope"]
    .value_counts()
    .rename_axis("pair_scope")
    .reset_index(name="pairs")
)
display(pair_scope_stats)


## Baseline score distribution

`baseline_similarity_score` — это dependency-free baseline в диапазоне 0..1: token Jaccard/containment + `SequenceMatcher` по нормализованному title. Это не финальный matcher, а retrieval baseline для gold-set разметки.


In [ ]:
score_summary = candidates_to_save["baseline_similarity_score"].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
).to_frame("baseline_similarity_score")
display(score_summary)

ax = candidates_to_save["baseline_similarity_score"].hist(bins=40, figsize=(10, 4), color="#4C78A8")
ax.set_title("Distribution of baseline_similarity_score")
ax.set_xlabel("baseline_similarity_score")
ax.set_ylabel("candidate pairs")
plt.tight_layout()
plt.show()


## Hard-negative sanity check

Hard-negative candidate здесь означает: brand совпадает, weight/pack примерно совпадает, но найденные вкусовые/типовые токены различаются. Это эвристика для разметки, не автоматическая метка `different_product`.


In [ ]:
hard_negative_stats = candidates_to_save["is_hard_negative_candidate"].value_counts(dropna=False).rename_axis("is_hard_negative_candidate").reset_index(name="pairs")
display(hard_negative_stats)

display(
    candidates_to_save[candidates_to_save["is_hard_negative_candidate"]]
    .sort_values("baseline_similarity_score", ascending=False)
    .head(10)
)


## Sanity-check examples

Смотрим несколько пар из разных зон score. Эти примеры нужны только для проверки, что baseline выдаёт осмысленные пары перед ручной разметкой.


In [ ]:
def show_examples(frame: pd.DataFrame, label: str, n: int = 5) -> None:
    if frame.empty:
        print(f"{label}: нет примеров")
        return
    print(label)
    display(
        frame[[
            "raw_record_id_a",
            "raw_record_id_b",
            "marketplace_a",
            "marketplace_b",
            "sku_a",
            "sku_b",
            "title_a",
            "title_b",
            "brand_a",
            "brand_b",
            "unit_amount_a",
            "unit_amount_b",
            "total_amount_a",
            "total_amount_b",
            "multipack_count_a",
            "multipack_count_b",
            "baseline_similarity_score",
            "is_cross_marketplace_pair",
            "is_hard_negative_candidate",
        ]].head(n)
    )

show_examples(candidates_to_save[candidates_to_save["baseline_similarity_score"] >= 0.85], "High similarity")
show_examples(
    candidates_to_save[
        (candidates_to_save["baseline_similarity_score"] >= 0.55)
        & (candidates_to_save["baseline_similarity_score"] < 0.72)
    ],
    "Medium similarity",
)
show_examples(candidates_to_save[candidates_to_save["is_cross_marketplace_pair"]], "Cross-marketplace candidates")
show_examples(candidates_to_save[candidates_to_save["is_hard_negative_candidate"]], "Hard-negative candidates")


## Итог

Главный артефакт этого ноутбука — `research/dedup/data/candidates_sauces.csv`. Следующий ноутбук использует его для стратифицированной выборки под ручную разметку.
